In [3]:
!pip install -r ../requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 777.2 kB/s eta 0:00:0000:0100:01


In [3]:
!python -m spacy download ru_core_news_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.4/513.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 99.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
from natasha import (
    Segmenter, MorphVocab, NewsEmbedding,
    NewsMorphTagger, Doc
)
from nltk.corpus import stopwords
from nltk import download as nltk_download
import string

nltk_download('stopwords')

segmenter = Segmenter()
morph_vocab = MorphVocab()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)

russian_stopwords = set(stopwords.words('russian'))

def read_conll(filepath):
    sentences = []
    sentence = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                if sentence:
                    sentences.append(sentence)
                    sentence = []
            else:
                parts = line.split()
                doc = Doc(parts[0])
                doc.segment(segmenter)
                for token in doc.tokens:
                  sentence.append([token.text, parts[1]])
        if sentence:
            sentences.append(sentence)
    return sentences

def lemmatize_sentences(sentences):
  result_sentences = []
  for sentence in sentences:
    text = ' '.join(tuple(zip(*sentence))[0])
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)

    lemmatized_tokens = []

    for idx, token in enumerate(doc.tokens):
        token.lemmatize(morph_vocab)
        lemmatized_tokens.append([token.lemma.lower(), sentence[idx][1]])

    result_sentences.append(lemmatized_tokens)
  return result_sentences

def write_conll(sentences, out_path):
  with open(out_path, "w", encoding="utf-8") as f:
    for sent in sentences:
      for token in sent:
        f.write(f"{token[0]} {token[1]}\n")
      f.write("\n")

def lemmatize_conll(input_path, output_path):
  init_sentences = read_conll(input_path)
  lemmatized_sentences = lemmatize_sentences(init_sentences)
  write_conll(lemmatized_sentences, output_path)


[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [5]:
lemmatize_conll('./init dataset/all_ner_test.tsv', './data/all_ner_test_lemma.tsv')
lemmatize_conll('./init dataset/all_ner_train.tsv', './data/all_ner_train_lemma.tsv')

In [6]:
!python -m spacy convert ./data/all_ner_test_lemma.tsv ./data -t json -n 1 -c iob -l ru
!python -m spacy convert ./data/all_ner_train_lemma.tsv ./data -t json -n 1 -c iob -l ru

ℹ Auto-detected token-per-line NER format
ℹ Grouping every 1 sentences into a document.
⚠ To generate better training data, you may want to group sentences
into documents with `-n 10`.
✔ Generated output file (1 documents): data/all_ner_test_lemma.json
ℹ Auto-detected token-per-line NER format
ℹ Grouping every 1 sentences into a document.
⚠ To generate better training data, you may want to group sentences
into documents with `-n 10`.
✔ Generated output file (1 documents):
data/all_ner_train_lemma.json


In [7]:
!python -m spacy convert ./data/all_ner_test_lemma.json ./data -t spacy
!python -m spacy convert ./data/all_ner_train_lemma.json ./data -t spacy

✔ Generated output file (259 documents):
data/all_ner_test_lemma.spacy
✔ Generated output file (2322 documents):
data/all_ner_train_lemma.spacy


In [8]:
!python -m spacy debug data ./rubert-tiny2_spacy_config.cfg --paths.train ./data/all_ner_train.spacy --paths.dev ./data/all_ner_test.spacy --code ./custom_factory.py


============================ Data file validation ============================
✔ Pipeline can be initialized with data
✔ Corpus is loadable

=============================== Training stats ===============================
Language: ru
Training pipeline: transformer, ner
2322 training docs
259 evaluation docs
✔ No overlap between training and evaluation data

============================== Vocab & Vectors ==============================
ℹ 290812 total word(s) in the data (25120 unique)
ℹ No word vectors present in the package

================================== Summary ==================================
✔ 3 checks passed


In [9]:
!python -m spacy train "./rubert-tiny2_spacy_config.cfg" --paths.train "./data/all_ner_train_lemma.spacy" --paths.dev "./data/all_ner_test_lemma.spacy" --output ./training --gpu-id -1 --code "./custom_factory.py"

ℹ Saving to output directory: training
ℹ Using CPU

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'ner']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS...  LOSS NER  F1_MICRO  F1_MACRO  F1_WEIGHTED  F1_COMPONENT  F1_SYSTEM  F1_ATTRIBUTE  ENTS_P  ENTS_R  ENTS_F  SCORE 
---  ------  -------------  --------  --------  --------  -----------  ------------  ---------  ------------  ------  ------  ------  ------
  0       0        1068.22    636.56      0.31      0.23         0.24          0.28       0.40          0.00    0.29    0.33    0.31    0.00
  0      50       68615.59  35464.63      0.04      0.08         0.02          0.00       0.23          0.00    0.16    0.02    0.04    0.00
  0     100       28195.80  17626.36      0.00      0.00         0.00          0.00       0.00          0.00    0.00    0.00    0.00    0.00
  

In [ ]:
import spacy
from spacy.pipeline.ner import EntityRecognizer
from spacy.language import Language
from thinc.api import Config
from sklearn.metrics import f1_score, precision_recall_fscore_support
import plotly.express as px
import json
import os
from pathlib import Path

from custom_factory import *

nlp = spacy.load("./training/model-best")

In [ ]:
text = [
    "Вставка декоративная обивки двери салона автомобиля удлиненного прямолинейного контура коробообразной формы поперечного сечения, выполненная с возможностью фиксирования на обивке двери и креплением на ней устройства подсветки в сборе посредством крепежно-фиксирующих элементов, конструктивно выполненных в виде зацепов, приварных элементов, позиционирующих пинов, ответных отверстий и клипс, причем приварные элементы, равномерно сформированные по контуру тыльной стороны вставки декоративной, входят в зацепление с корпусом устройства подсветки через ответные отверстия, жестко соединяя их методом сварки, а зацепы, равноудаленно сформированные на наружной поверхности корпуса устройства подсветки и выполненные с усилительными ребрами по боковым кромкам на переднем участке, разнонаправленными под углом друг к другу и основной поверхности зацепов, входят в зацепление с ответными клипсами, предварительно установленными в поверхности соответствующей зоны обивки двери, эквидистантно расположению и количеству зацепов сопрягаемой поверхности, позиционируя и ограничивая перемещение декоративной вставки относительно обивки двери, позиционирующие пины, выполненные четырехгранными и расположенными на концевых участках наружной поверхности корпуса устройства подсветки, позиционируют декоративную вставку на обивке двери при монтаже, позиционирующий элемент типа «гарпун», расположенный по центру наружной поверхности корпуса устройства подсветки, фиксирует декоративную вставку на обивке двери, ограничивая перемещение декоративной вставки в направлении «на вылет» из посадочных мест."
]

for doc in nlp.pipe(text, disable=["tagger", "parser"]):
  print([(ent.text, ent.label_) for ent in doc.ents])

[('Вставка декоративная обивки двери салона автомобиля удлиненного прямолинейного контура', 'SYSTEM'), ('креплением', 'COMPONENT'), ('устройства подсветки', 'COMPONENT'), ('крепежно-фиксирующих элементов', 'COMPONENT'), ('конструктивно выполненных в виде зацепов', 'ATTRIBUTE'), ('приварных элементов', 'COMPONENT'), ('пинов', 'COMPONENT'), ('ответных отверстий', 'COMPONENT'), ('клипс', 'COMPONENT'), ('приварные элементы', 'COMPONENT'), ('корпусом устройства подсветки', 'COMPONENT'), ('ответные отверстия', 'COMPONENT'), ('зацепы', 'COMPONENT'), ('наружной поверхности', 'COMPONENT'), ('устройства подсветки', 'COMPONENT'), ('усилительными ребрами', 'COMPONENT'), ('боковым кромкам', 'COMPONENT'), ('основной поверхности зацепов', 'COMPONENT'), ('ответными клипсами', 'COMPONENT'), ('зоны обивки двери', 'COMPONENT'), ('декоративной вставки', 'COMPONENT'), ('двери', 'COMPONENT'), ('позиционирующие пины', 'COMPONENT'), ('четырехгранными', 'ATTRIBUTE'), ('концевых участках', 'COMPONENT'), ('наруж